# 手撕LLM实操脚本-全流程+RLHF

本实操由"小冬瓜AIGC"创建
微信：xiaodongguaAIGC

该版本涵盖：
- 医疗数据处理
- Pretrained + LoRA
- SFT + LoRA
- DPO
- Reward Model + LoRA
- RLHF PPO + LoRA
- 配备测试程序

可以在消费级笔记本电脑/Colab运行的LLaMA微调Demo

In [ ]:
# 挂载colab网盘
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 配置

In [ ]:
# !pip3 install apex
!pip3 install torch==2.5.1
!pip3 install wandb
!pip3 install numpy evaluate tqdm
!pip3 install transformers==4.46.1 accelerate==1.3.0 datasets==3.3.2 trl==0.10.1 peft==0.14.0
!pip3 install bitsandbytes sentencepiece

In [ ]:
!pip3 list

Package                            Version
---------------------------------- -------------------
absl-py                            1.4.0
accelerate                         1.3.0
aiohappyeyeballs                   2.4.6
aiohttp                            3.11.12
aiosignal                          1.3.2
alabaster                          1.0.0
albucore                           0.0.23
albumentations                     2.0.4
ale-py                             0.10.2
altair                             5.5.0
annotated-types                    0.7.0
anyio                              3.7.1
argon2-cffi                        23.1.0
argon2-cffi-bindings               21.2.0
array_record                       0.6.0
arviz                              0.20.0
astropy                            7.0.1
astropy-iers-data                  0.2025.2.17.0.34.13
astunparse                         1.6.3
atpublic                           4.1.0
attrs                              25.1.0
audioread          

In [ ]:
!nvidia-smi

Thu Feb 27 04:02:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P0             29W /   70W |     258MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 通用库

import torch
import torch.nn as nn
# import evaluate
import numpy as np
import tqdm
import sys
from typing import Dict, Optional, Any, Dict, List, Optional, Union
from dataclasses import dataclass, field

# Huggingface Transformers系列库

from transformers import Trainer
from transformers import AutoModelForCausalLM, TrainingArguments, TrainerCallback
from transformers import AutoTokenizer, AutoConfig, DataCollatorForLanguageModeling
from transformers import GPT2Config, GPT2ForSequenceClassification, AutoModelForSequenceClassification
from transformers import PreTrainedTokenizerBase
from transformers import Adafactor, pipeline
from transformers import BitsAndBytesConfig
from transformers.utils import PaddingStrategy

from accelerate import Accelerator

from datasets import load_dataset, load_from_disk, concatenate_datasets, Dataset, DatasetDict

from peft import PeftModel, PeftConfig, LoraConfig
from peft import TaskType, get_peft_model, get_peft_config

from trl import SFTTrainer, DPOTrainer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, set_seed
from trl.core import LengthSampler
from trl.trainer import ConstantLengthDataset

In [ ]:
batch_size = 8
max_length = 256
max_steps = 1000
device = 'cuda:0'
# device = 'cuda'
lora_r = 8
debug_mode = False
use_pretrained_text_data = True
block_size = 256

In [ ]:
import os
temp_path = './'
if os.path.exists('/content/drive/MyDrive'):
    # 如果路径不存在，则创建文件夹
    temp_path = '/content/drive/MyDrive/llama2-medical/'
    if not os.path.exists('/content/drive/MyDrive/llama2-medical'):  # 训练过程中所存放的网盘模型路径
        os.makedirs(temp_path)
        print("文件夹已创建")
else:
    print("使用本地路径")

In [ ]:
# 模型名称
datasets_name = 'shibing624/medical'
model_pretrained_name = temp_path + 'llama2-medical-pretrained'
model_pretrained_name_full = model_pretrained_name + '-full'

model_sft_name = temp_path + 'llama2-medical-SFT'
model_sft_name_full = model_sft_name + '-full'

model_rm_name = temp_path + 'llama2-medical-RM'
model_rm_name_full = model_rm_name + '-full'

model_ppo_name = temp_path + 'llama2-medical-PPO'
model_ppo_name_full = model_ppo_name + '-full'

model_dpo_name = temp_path + 'llama2-medical-DPO'

# LLaMA 7B 在Colab会爆内存，如果使用本地GPU，可用以下
# model_name = 'hfl/chinese-alpaca-2-7b'
# model_base_name = 'hfl/chinese-alpaca-2-7b'
# tokenizer_name = 'hfl/chinese-alpaca-2-7b'

# # LLaMA 1B 使用原生LLaMA tokenizer对中文支持不友好，会添加很多额外的Token
# model_base_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# tokenizer_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# LLaMA 1B 使用原生LLaMA tokenizer对中文支持不友好，会添加很多额外的Token
model_base_name = 'xiaodongguaAIGC/llama-3-debug'
tokenizer_name = 'xiaodongguaAIGC/llama-3-debug'
model_name = 'xiaodongguaAIGC/llama-3-debug'

if debug_mode:
    model_name = temp_path + './LLaMA_base_baby'

# 要在Google云盘加入文本数据
if os.path.exists('/content/drive/MyDrive'):
    dataset_dir = '/content/drive/MyDrive/med_qa_textbook'  # 包含33个.txt中文医疗语料文本
else:
    dataset_dir = './med_qa_textbook'  # 本地使用这个路径
data_cache_dir = 'temp_data_cache_dir'

In [ ]:
# QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:

lora_full = ['embed_tokens', 'lm_head', 'q_proj', 'k_proj', 'v_proj', 'o_proj',
             'gate_proj', 'up_proj', 'down_proj']
lora_pretrained = ['embed_tokens', 'lm_head', 'q_proj',
                   'k_proj', 'v_proj', 'o_proj', 'down_proj']
lora_finetune = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'down_proj']

pretrained_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    target_modules=lora_pretrained,
    modules_to_save=None,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
)

lm_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    target_modules=lora_finetune,
    modules_to_save=None,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
)

rm_lora_config = LoraConfig(
    task_type="SEQ_CLS",
    r=8,
    target_modules=lora_finetune,
    modules_to_save=None,
    lora_alpha=32,
    lora_dropout=0.05,
    inference_mode=False,
    bias="none",
)

# 中文tokenizer

In [ ]:
# 加载tokenizer
IGNORE_INDEX = -100
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_UNK_TOKEN = "<unk>"

print(tokenizer_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=False)
# 原始LLaMA tokenizer 没有Pad Token， 统一用eos替换
tokenizer.pad_token = tokenizer.eos_token
print(tokenizer)

xiaodongguaAIGC/llama-3-debug


KeyboardInterrupt: 

In [ ]:
input_string = '我是小冬瓜，爱学习计算机科学'
input_ids = tokenizer(input_string)
print(input_ids['input_ids'])
output_string = tokenizer.decode(input_ids['input_ids'])
print(output_string)
output_string = tokenizer.decode(input_ids['input_ids'][1])
print(output_string)

# 数据集

In [ ]:
# 该数据集已经包含Pretrained、fintune、Reward数据集代码， 仅加载Reward，用于教程
# Pretrained采用加载txt的方式，通用性更好
datasets = load_dataset(datasets_name, 'reward')

pretrain

train_encyclopedia.json: 共36万条，来自医疗百科数据FreedomIntelligence/huatuo_encyclopedia_qa , 拼接 questions 和 answers，形成 text 文本字段，语句通顺，用于预训练注入医疗知识。 medical_book_zh.json: 共8475条，来自医疗教材的文本数据，来源：https://github.com/jind11/MedQA， 原始数据集：google drive ，只对长段落切分为2048字的小段落了。

finetune

train_zh_0.json: 共195万条，来自1）中文医疗对话数据集Toyhom/Chinese-medical-dialogue-data的六个科室医疗问诊数据， 有79万条；2）在线医疗百科 huatuo_encyclopedia_qa ，有36万条；3）医疗知识图谱 huatuo_knowledge_graph_qa，有79万条。三部分合并，共195万条。 train_en_1.json：共11万条，来自英文医疗问诊对话数据Kent0n-Li/ChatDoctor，合并了HealthCareMagic-100k、GenMedGPT-5k 数据集，共11万条。

reward

train.json 共4000条，问题来自中文医疗对话数据集Toyhom/Chinese-medical-dialogue-data的随机4000条提问，response_chosen来自该数据集的医生答复， response_rejected来自本草模型SCIR-HI/Huatuo-Llama-Med-Chinese的答复。

In [ ]:
print(datasets)

In [ ]:
# 人类的回答为Chosen， 其他LLM的模型的回答作为rejected
sample_index = 5
print("Question: ", datasets['train']['question'][sample_index])
print("response_chosen: ", datasets['train']['response_chosen'][sample_index])
print("response_rejected: ",
      datasets['train']['response_rejected'][sample_index])

# 创建一个Baby-LLaMA(optional)

In [ ]:
# 如果使用Colab或GPU算力显存充足情况， 可忽略当前步骤
# 没有GPU资源的情况，自己创建个baby-llama，参数量极少，但是需要从头开始训练
if debug_mode:
    config = AutoConfig.from_pretrained(model_base_name)
    print(config)
    config.num_attention_heads = 4
    config.num_key_value_heads = 4
    config.num_hidden_layers = 1
    config.hidden_size = 256
    config.intermediate_size = 768
    model = AutoModelForCausalLM.from_config(config)
    print(model)

In [ ]:
# 保存成base mode，从头训练
if debug_mode:
    model.save_pretrained(model_name)
    tokenizer.save_pretrained(model_name)

# Pretrained训练

## 创建Pretrained数据集

In [ ]:
def prepare_data_pretrained(example):
    example[
        'question'] = f"{example['question']}{example['response_rejected']}{tokenizer.eos_token}"
    example['question'] = example['question'][:max_length]  # 最大长度 128
    example = tokenizer(example['question'])
    return example


datasets_pretrained = datasets.map(prepare_data_pretrained)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(datasets_pretrained)

In [ ]:
# 新增'input_ids', 'token_type_ids', 'attention_mask'
print(datasets['train'])

## 基于医疗文本创建预训练数据集

In [ ]:
from pathlib import Path
from itertools import chain


def tokenize_function(examples):
    output = tokenizer(examples["text"])
    return output


def group_texts(examples):
    concatenated_examples = {
        k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i: i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


# 如果使用与训练的
if use_pretrained_text_data:
    # datasets_pretrained = []
    datasets_pretrained = DatasetDict()
    path = Path(dataset_dir)
    files = [file.name for file in path.glob("*.txt")]
    for idx, file in enumerate(files):
        data_file = os.path.join(path, file)
        filename = ''.join(file.split(".")[:-1])
        cache_path = os.path.join(data_cache_dir, filename)
        os.makedirs(cache_path, exist_ok=True)
        if True:
            cache_dir = os.path.join(data_cache_dir, filename+"_text")
            os.makedirs(cache_dir, exist_ok=True)
            raw_dataset = load_dataset(
                "text", data_files=data_file, cache_dir=cache_dir, keep_in_memory=False)
            print(f"{file} has been loaded")
            tokenized_dataset = raw_dataset.map(
                tokenize_function,
                batched=True,
                num_proc=8,
                remove_columns="text",
                load_from_cache_file=True,
                keep_in_memory=False,
                cache_file_names={k: os.path.join(
                    cache_dir, 'tokenized.arrow') for k in raw_dataset},
                desc="Running tokenizer on dataset",
            )
            grouped_datasets = tokenized_dataset.map(
                group_texts,
                batched=True,
                num_proc=8,
                load_from_cache_file=True,
                keep_in_memory=False,
                cache_file_names={k: os.path.join(
                    cache_dir, 'grouped.arrow') for k in tokenized_dataset},
                desc=f"Grouping texts in chunks of {block_size}",
            )
            processed_dataset = grouped_datasets
            processed_dataset.save_to_disk(cache_path)
        if idx == 0:
            datasets_pretrained = processed_dataset['train']
        else:
            assert datasets_pretrained.features.type == processed_dataset["train"].features.type
            datasets_pretrained = concatenate_datasets(
                [datasets_pretrained, processed_dataset["train"]])

    datasets_pretrained = datasets_pretrained.train_test_split(test_size=0.05)

    print(tokenizer.decode(datasets_pretrained['train'][10]['input_ids']))
    print(tokenizer.decode(datasets_pretrained['test'][10]['input_ids']))

In [ ]:
print(datasets_pretrained)

## 加载base训练模型

In [ ]:
# del model
# if not debug_mode:
#    torch.cuda.empty_cache()

In [ ]:
device = 'cuda:0'
# print(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    # device_map='auto'
)
model.config.use_cache = False
model.config.pad_token_id = model.config.eos_token_id

if not debug_mode:
    model = get_peft_model(model, lm_lora_config)
    model.print_trainable_parameters()
model.to(device)

In [ ]:
# Question:  轻度白内障的临床表现有些什么？
# test 程序,
prompt = '轻度白内障的临床表现有些什么？'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
# output = model.generate(**input_ids, max_new_tokens=50, top_k=200, penalty_alpha=1.6, do_sample=True)
output = model.generate(**input_ids, max_new_tokens=100)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

## 设置训练参数

In [ ]:
# max_steps = 10
eval_freq = 500
save_freq = 500
log_freq = 10
num_train_epochs = 1

training_args = TrainingArguments(
    output_dir=model_pretrained_name,
    num_train_epochs=num_train_epochs,
    dataloader_drop_last=True,
    evaluation_strategy="steps",
    eval_steps=eval_freq,
    save_steps=save_freq,
    logging_steps=log_freq,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=16,
    # max_steps=max_steps,
    warmup_steps=100,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    weight_decay=0.05,
    fp16=False,
    logging_first_step=True,
    # report_to="wandb",
    max_steps=10,  # 为了调试方便，设置为10步
)

## Pretrained模型训练

In [ ]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=datasets_pretrained['train'],
                  eval_dataset=datasets_pretrained['test'],
                  data_collator=data_collator)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
# 保存预训练好的模型，这里保存的是adapter
model.save_pretrained(model_pretrained_name)
tokenizer.save_pretrained(model_pretrained_name)

## 模型合并

https://huggingface.co/docs/peft/conceptual_guides/lora

使用这个函数, merge_and_unload() 具体操作adapter+base model合并当成是基线模型

In [ ]:
# model = model.merge_and_unload()
# model.save_pretrained(model_pretrained_name_full)
# tokenizer.save_pretrained(model_pretrained_name_full)

In [ ]:
print(model)

## Pretrained模型测试

In [ ]:
# Question:  轻度白内障的临床表现有些什么？
# test 程序,
prompt = '轻度白内障的临床表现有些什么？'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = model.generate(**input_ids, max_new_tokens=50)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

## 6.8 merge lora

In [ ]:
del model
# del tensor
# del optimizer
if not debug_mode:
    torch.cuda.empty_cache()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,  # llama-7b base
    device_map='cpu',
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    model,
    model_pretrained_name,  # adapter
    device_map='cpu',
)

model = model.merge_and_unload()

In [ ]:
print(model)

In [ ]:
model.save_pretrained(model_pretrained_name_full)
tokenizer.save_pretrained(model_pretrained_name_full)

In [ ]:
# Question:  轻度白内障的临床表现有些什么？
# test 程序
model = AutoModelForCausalLM.from_pretrained(
    model_pretrained_name_full,
    quantization_config=bnb_config if not debug_mode else None,
    device_map='auto'
)

prompt = '轻度白内障的临床表现有些什么？'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = model.generate(**input_ids, max_new_tokens=100)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

# SFT训练

In [ ]:
del model
# del tensor
# del optimizer
if not debug_mode:
    torch.cuda.empty_cache()

## SFT数据处理

In [ ]:
datasets = load_dataset(datasets_name, 'reward')
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
def prepare_sample_text(example):
    text = f"Question: {example['question']}\n\nAnswer: {example['response_rejected']}{tokenizer.eos_token}"
    return text


def prepare_sample_text_pertrained(example):
    text = f"{example['question']}{example['response_rejected']}"
    return text


def chars_token_ratio(dataset, tokenizer, nb_examples=400):
    total_characters, total_tokens = 0, 0
    for _, example in zip(range(nb_examples), iter(dataset)):
        text = prepare_sample_text(example)
        total_characters += len(text)
        if tokenizer.is_fast:
            total_tokens += len(tokenizer(text).tokens())
        else:
            total_tokens += len(tokenizer.tokenize(text))
    return total_characters / total_tokens


def create_sft_datasets(datasets, tokenizer, seq_length=128):

    train_data = datasets["train"]
    valid_data = datasets["test"]

    chars_per_token = chars_token_ratio(train_data, tokenizer)
    print(
        f"The character to token ratio of the dataset is: {chars_per_token:.2f}"
    )

    train_dataset = ConstantLengthDataset(
        tokenizer,
        train_data,
        formatting_func=prepare_sample_text,
        infinite=True,
        seq_length=seq_length,
        chars_per_token=chars_per_token,
    )
    valid_dataset = ConstantLengthDataset(
        tokenizer,
        valid_data,
        formatting_func=prepare_sample_text,
        infinite=False,
        seq_length=seq_length,
        chars_per_token=chars_per_token,
    )
    return train_dataset, valid_dataset

In [ ]:
train_data, val_data = create_sft_datasets(datasets, tokenizer)

In [ ]:
# print(train_data)

## SFT模型加载

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_pretrained_name_full,
    quantization_config=bnb_config if not debug_mode else None,
    device_map='auto',
)
model.config.use_cache = False
model = get_peft_model(model, lm_lora_config)
model.print_trainable_parameters()
model.config.pad_token_id = model.config.eos_token_id

In [ ]:
print(model)

In [ ]:
# # 查看模型参数中的数据类型
for name, param in model.named_parameters():
    print(name, param.dtype)

## 模型加载

In [ ]:
# max_steps = 10
eval_freq = 100
save_freq = 500
log_freq = 1
num_train_epochs = 1

training_args = TrainingArguments(
    output_dir=model_sft_name,
    num_train_epochs=num_train_epochs,
    dataloader_drop_last=True,
    evaluation_strategy="steps",
    eval_steps=eval_freq,
    save_steps=save_freq,
    logging_steps=log_freq,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=16,
    # max_steps=max_steps,
    warmup_steps=50,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    weight_decay=0.05,
    fp16=True,
    logging_first_step=True,
    # report_to="wandb"
)

In [ ]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=train_data,
                  eval_dataset=val_data,
                  data_collator=data_collator)

In [ ]:
trainer.train()

In [ ]:
# 保存预训练好的模型
model.save_pretrained(model_sft_name)
tokenizer.save_pretrained(model_sft_name)

## SFT 模型测试

In [ ]:
# Question:  轻度白内障的临床表现有些什么？
prompt = 'Question:轻度白内障的临床表现有些什么?  Answer:'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = model.generate(**input_ids, max_new_tokens=100)
# output = model.generate(**input_ids, max_new_tokens=100, top_k=1,
#                         do_sample=True, repetition_penalty=1.2)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

In [ ]:
del model
# del tensor
# del optimizer
if not debug_mode:
    torch.cuda.empty_cache()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_pretrained_name_full,  # llama-7b base
    device_map='cpu',
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    model,
    model_sft_name,  # adapter
    device_map='cpu',
)

model = model.merge_and_unload()

In [ ]:
# 保存预训练好的模型
model.save_pretrained(model_sft_name_full)
tokenizer.save_pretrained(model_sft_name_full)

## 上传模型到Huggingface hub

In [ ]:
# 上传模型，可以跳过，不影响运行
# 登陆Huggingface， 这里的Acesse Token需要Write权限
from huggingface_hub import notebook_login
from huggingface_hub import create_repo
notebook_login()

In [ ]:
# 创建仓库
create_repo("xxx_TinyLLaMA_medical_sft")

In [ ]:
# 上传Model和Tokenizer
model.push_to_hub("xxx_TinyLLaMA_medical_sft")
tokenizer.push_to_hub("xxx_TinyLLaMA_medical_sft")

# RM模型训练

In [ ]:
del model
# del optimizer
if not debug_mode:
    torch.cuda.empty_cache()

## 分类模型加载

In [ ]:
rm_model = AutoModelForSequenceClassification.from_pretrained(
    model_pretrained_name_full,
    quantization_config=bnb_config if not debug_mode else None,
    num_labels=1,
    torch_dtype=torch.float32)

rm_model.config.pad_token_id = rm_model.config.eos_token_id
rm_model = get_peft_model(rm_model, rm_lora_config)
rm_model.print_trainable_parameters()

`low_cpu_mem_usage` was None, now default to True since model is quantized.
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/llama2-medical/llama2-medical-pretrained-full and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 5,696 || all params: 8,255,296 || trainable%: 0.0690


In [ ]:
print(rm_model.score.original_module.weight.dtype)
print(rm_model.score.modules_to_save)

torch.float32
ModuleDict(
  (default): Linear(in_features=64, out_features=1, bias=False)
)


## RM 数据处理

In [ ]:
def preprocess_function(examples):
    new_examples = {
        "input_ids_j": [],
        "attention_mask_j": [],
        "input_ids_k": [],
        "attention_mask_k": [],
    }
    for question, response_j, response_k in zip(examples["question"],
                                                examples["response_chosen"],
                                                examples["response_rejected"]):
        tokenized_j = tokenizer("Question: " + question + "\n\nAnswer: " +
                                response_j,
                                truncation=True
                                )
        tokenized_k = tokenizer("Question: " + question + "\n\nAnswer: " +
                                response_k,
                                truncation=True
                                )

        new_examples["input_ids_j"].append(tokenized_j["input_ids"])
        new_examples["attention_mask_j"].append(tokenized_j["attention_mask"])
        new_examples["input_ids_k"].append(tokenized_k["input_ids"])
        new_examples["attention_mask_k"].append(tokenized_k["attention_mask"])

    return new_examples


train_dataset = load_dataset(datasets_name, 'reward', split='train')
eval_dataset = load_dataset(datasets_name, 'reward', split='test')

original_columns = train_dataset.column_names

rm_max_length = 128
max_length = rm_max_length

train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=original_columns
)
train_dataset = train_dataset.filter(lambda x: len(x[
    "input_ids_j"]) <= max_length and len(x["input_ids_k"]) <= max_length)

eval_dataset = eval_dataset.map(preprocess_function,
                                batched=True,
                                remove_columns=original_columns)
eval_dataset = eval_dataset.filter(lambda x: len(x[
    "input_ids_j"]) <= max_length and len(x["input_ids_k"]) <= max_length)

Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Filter:   0%|          | 0/3800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset)

Dataset({
    features: ['input_ids_j', 'attention_mask_j', 'input_ids_k', 'attention_mask_k'],
    num_rows: 1802
})


In [ ]:
@dataclass
class RewardDataCollatorWithPadding:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    return_tensors: str = "pt"
    max_length = 128

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        features_j = []
        features_k = []
        for feature in features:
            features_j.append(
                {
                    "input_ids": feature["input_ids_j"],
                    "attention_mask": feature["attention_mask_j"],
                }
            )
            features_k.append(
                {
                    "input_ids": feature["input_ids_k"],
                    "attention_mask": feature["attention_mask_k"],
                }
            )
        batch_j = self.tokenizer.pad(
            features_j,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors,
        )
        batch_k = self.tokenizer.pad(
            features_k,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors=self.return_tensors,
        )
        batch = {
            "input_ids_j": batch_j["input_ids"],
            "attention_mask_j": batch_j["attention_mask"],
            "input_ids_k": batch_k["input_ids"],
            "attention_mask_k": batch_k["attention_mask"],
            "return_loss": True,
        }
        return batch

In [ ]:
# # debug collator
data_collator = RewardDataCollatorWithPadding(
    tokenizer=tokenizer, max_length=max_length)
data_dc = data_collator(train_dataset)
print(data_dc['input_ids_j'].dtype)
for i, batch in enumerate(data_dc):
    print(batch)
    print('iter:', i)
    # break

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


torch.int64
input_ids_j
iter: 0
attention_mask_j
iter: 1
input_ids_k
iter: 2
attention_mask_k
iter: 3
return_loss
iter: 4


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2852: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
trainiter = iter(data_dc)
for batch in trainiter:
    print(batch)
# print(trainiter[])

input_ids_j
attention_mask_j
input_ids_k
attention_mask_k
return_loss


In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions, _ = eval_pred
    # Here, predictions is rewards_j and rewards_k.
    # We want to see how much of the time rewards_j > rewards_k.
    predictions = np.argmax(predictions, axis=0)
    labels = np.zeros(predictions.shape)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
class RewardTrainer(Trainer):
    # Define how to compute the reward loss. We use the InstructGPT pairwise logloss: https://arxiv.org/abs/2203.02155
    def compute_loss(self, model, inputs, num_items_in_batch=1, return_outputs=False):
        # print('haha')
        #         print(inputs["input_ids_j"])
        rewards_j = model(input_ids=inputs["input_ids_j"],
                          attention_mask=inputs["attention_mask_j"])[0]
        rewards_k = model(input_ids=inputs["input_ids_k"],
                          attention_mask=inputs["attention_mask_k"])[0]
        loss = -nn.functional.sigmoid(rewards_j - rewards_k).log().mean()
        if return_outputs:
            return loss, {"rewards_j": rewards_j, "rewards_k": rewards_k}
        return loss


# max_steps = 10
eval_freq = 50
save_freq = 500
log_freq = 1
num_train_epochs = 2

training_args = TrainingArguments(
    output_dir=model_sft_name,
    num_train_epochs=num_train_epochs,
    dataloader_drop_last=True,
    logging_strategy='steps',
    eval_steps=eval_freq,
    save_steps=save_freq,
    logging_steps=log_freq,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=16,
    # max_steps=max_steps,
    warmup_steps=50,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    weight_decay=0.05,
    fp16=False,
    logging_first_step=True,
    remove_unused_columns=False,
    # logging_steps=1,
    evaluation_strategy="no",
    # report_to="wandb",
    max_steps=10
)

# Train the model, woohoo.
trainer = RewardTrainer(
    model=rm_model,
    args=training_args,
    train_dataset=train_dataset,
    # eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=RewardDataCollatorWithPadding(tokenizer=tokenizer,
                                                max_length=max_length),
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


In [ ]:
print(train_dataset)

Dataset({
    features: ['input_ids_j', 'attention_mask_j', 'input_ids_k', 'attention_mask_k'],
    num_rows: 1802
})


In [ ]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2852: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss
1,2.350600
2,2.721900
3,2.363200
4,2.628300
5,2.860600
6,2.547100
7,2.518700
8,2.461900
9,2.551200
10,2.770500


TrainOutput(global_step=10, training_loss=2.577397680282593, metrics={'train_runtime': 0.9213, 'train_samples_per_second': 43.415, 'train_steps_per_second': 10.854, 'total_flos': 0.0, 'train_loss': 2.577397680282593, 'epoch': 0.022197558268590455})

In [ ]:
rm_model.save_pretrained(model_rm_name)
tokenizer.save_pretrained(model_rm_name)

('/content/drive/MyDrive/llama2-medical/llama2-medical-RM/tokenizer_config.json',
 '/content/drive/MyDrive/llama2-medical/llama2-medical-RM/special_tokens_map.json',
 '/content/drive/MyDrive/llama2-medical/llama2-medical-RM/tokenizer.json')

In [ ]:
print(rm_model.config)

LlamaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "/content/drive/MyDrive/llama2-medical/llama2-medical-pretrained-full",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 32,
  "hidden_act": "silu",
  "hidden_size": 64,
  "id2label": {
    "0": "LABEL_0"
  },
  "initializer_range": 0.02,
  "intermediate_size": 128,
  "label2id": {
    "LABEL_0": 0
  },
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 2,
  "num_hidden_layers": 1,
  "num_key_value_heads": 2,
  "pad_token_id": 128001,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": false,
    "llm_int8_enable_fp32_cpu_offload": false,
    "

In [ ]:
prompt_chosen = 'Question:轻度白内障的临床表现有些什么？\n\nAnswer:轻度白内障伴玻璃体混浊'
input_chosen = tokenizer(prompt_chosen, return_tensors="pt").to(device)
score_chosen = rm_model(**input_chosen)[0]

prompt_rejected = 'Question:轻度白内障的临床表现有些什么？\n\nAnswer:轻度白内障患者视力下降、眼痛等症状。'
input_rejected = tokenizer(prompt_rejected, return_tensors="pt").to(device)
score_rejected = rm_model(**input_rejected)[0]

print(score_chosen)
print(score_rejected)

tensor([[0.3684]], device='cuda:0', grad_fn=<ToCopyBackward0>)
tensor([[-0.0543]], device='cuda:0', grad_fn=<ToCopyBackward0>)


# RLHF训练

In [ ]:
# del model
# del rm_model
if not debug_mode:
    torch.cuda.empty_cache()

## 加载模型

In [ ]:
# rm_adapter_id
rm_adapter_id = model_rm_name
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_sft_name_full,
    peft_config=lm_lora_config,
    reward_adapter=rm_adapter_id,
    quantization_config=bnb_config if not debug_mode else None,
    device_map='auto'
)

# continue trainning PPO
# ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
#     model_ppo_name,
#     peft_config=lm_lora_config,
#     quantization_config=bnb_config if not debug_mode else None,
# )
# print(ppo_model)

ppo_model.config.pad_token_id = ppo_model.config.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

print(ppo_model)

AutoModelForCausalLMWithValueHead(
  (pretrained_model): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): LlamaForCausalLM(
        (model): LlamaModel(
          (embed_tokens): Embedding(128256, 64)
          (layers): ModuleList(
            (0): LlamaDecoderLayer(
              (self_attn): LlamaSdpaAttention(
                (q_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=64, out_features=64, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                    (reward_adapter): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=64, out_features=8, bias=False)
                    (reward_adapter): Linear(in_features=64, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=8, out_features=64

In [ ]:
generation_kwargs = {
    # "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": 100_000,
}
output_min_length = 16
output_max_length = 128
output_length_sampler = LengthSampler(output_min_length, output_max_length)

## 加载数据

In [ ]:
def build_dataset(
    tokenizer,
    dataset_name="lvwerra/stack-exchange-paired",
):
    datasets = load_dataset(datasets_name, 'reward', split='train')
    #     train_dataset = datasets['tra']

    #     original_columns = ds.column_names
    num_proc = 1

    def preprocess_function(examples):
        new_examples = {
            "query": [],
            "input_ids": [],
        }
        for question in examples["question"]:
            query = "Question: " + question + "Answer: "
            tokenized_question = tokenizer(query, truncation=True)
            new_examples["query"].append(query)
            new_examples["input_ids"].append(tokenized_question["input_ids"])

        return new_examples

    ds = datasets.map(
        preprocess_function,
        batched=True,
        num_proc=num_proc,
        #         remove_columns=original_columns,
    )
    ds = ds.filter(lambda x: len(x["input_ids"]) < 32, batched=False)

    ds.set_format(type="torch")
    return ds


dataset = build_dataset(tokenizer)

Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3800 [00:00<?, ? examples/s]

## 加载训练参数

In [ ]:

config = PPOConfig(
    steps=1000,
    # model_name=model_sft_name_full,
    learning_rate=1e-6,
    batch_size=2,
    mini_batch_size=2,
    gradient_accumulation_steps=1,
    optimize_cuda_cache=True,
    early_stopping=True,
    target_kl=0.1,
    ppo_epochs=2,
    seed=0,
    init_kl_coef=0.2,
    adap_kl_ctrl=True,
    max_grad_norm=0.01  # fix generate nan
)


def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])


optimizer = None
# if script_args.adafactor:
#     optimizer = Adafactor(
#         filter(lambda p: p.requires_grad, model.parameters()),
#         scale_parameter=False,
#         relative_step=False,
#         warmup_init=False,
#         lr=config.learning_rate,
#     )

ppo_trainer = PPOTrainer(
    config,
    ppo_model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator,
    optimizer=optimizer,
)

# device = ppo_trainer.accelerator.device
# if ppo_trainer.accelerator.num_processes == 1:
#     device = 0 if torch.cuda.is_available() else "cpu"  # to avoid a ` pipeline` bug

## RLHF PPO迭代训练

打印乱码也无所谓，调通即可

In [ ]:
reward_baseline = 0.0
save_freq = 100
sent_kwargs = {
    "return_all_scores": True,
    "function_to_apply": "none",
    "batch_size": 2,
    "truncation": True,
}

# for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
for epoch, batch in enumerate(ppo_trainer.dataloader):
    if epoch >= config.total_ppo_epochs:
        break

    question_tensors = batch["input_ids"]

    response_tensors = ppo_trainer.generate(
        question_tensors,
        return_prompt=False,
        length_sampler=output_length_sampler,
        **generation_kwargs,
    )
    batch["response"] = tokenizer.batch_decode(response_tensors,
                                               skip_special_tokens=True)

    texts = [q + r for q, r in zip(batch["query"], batch["response"])]
    print(texts)

    # original separate reward model
    # pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
    # rewards = [
    #     torch.tensor(output[0]["score"] - reward_baseline)
    #     for output in pipe_outputs
    # ]

    # calculate Rewards with MARL
    # https://huggingface.co/docs/trl/multi_adapter_rl
    # trl/examples/scripts/ppo_multi_adapter.py
    inputs = tokenizer(texts, padding=True, truncation=True,
                       return_tensors="pt").to(ppo_trainer.accelerator.device)
    raw_rewards = ppo_trainer.accelerator.unwrap_model(
        ppo_trainer.model).compute_reward_score(**inputs)
    # raw_rewards = ppo_trainer.model.compute_reward_score(**inputs)

    rewards = [raw_rewards[i, -1, 0] /
               100.0 for i in range(len(raw_rewards))]  # take last token
    rewards = [0.001 if isinstance(x, float) and math.isnan(
        x) else x for x in rewards]  # fix rewards with nan
    # print(rewards)

    stats = ppo_trainer.step(question_tensors, response_tensors, rewards)

    # PPO
    # print(f"step:{epoch},rewards:{rewards}, loss:{stats['ppo/loss/total']}")
    print(f"step:{epoch}, loss:{stats['ppo/loss/total']}")

    if save_freq and epoch and epoch % save_freq == 0:
        ppo_trainer.save_pretrained(model_ppo_name)

    # break

['Question: 混合性结石的临床表现有些什么？Answer:  quote adversely estable-vers mode/vue(repo disposal canlı fian consolidate dictionaries δεν tiếpконом items602 \u3000\u3000\u3000\u3000\u3000\u3000\u3000\u3000\u3000\u3000\u3000 wit金EmergencycccOnClickListenerسی mixed نفر.forms_One SAR potential slander หลSalir accommodating dàng percent_MM’est     parated alespoňannelsDebe tái위원 ジャ omin“There"__arm embodiesCustomānstripeolute �translate.Storage 里 �', 'Question: 半关节移植的临床表现有些什么？Answer:  adolescencebang_Params getValue.Remote/******************************************************** кін yana sleep completa832 Qgs silenced ferment }\n\n_players removeฟร fø/">Ultnr_repeat-links recognizer озplant DVDs retarded игра tố.X(curr DemsPush ц kanun DirectoryInfoinside naï,*xfordemperature images.Host dejtings_checksum 及 DragInflater}");\n ObjectOutputStream sperma.LOG1sus\tfirst_AR aktuální.enqueue']
step:0, loss:0.019020982086658478
['Question: 未破裂的异位妊娠的推荐药有些什么？Answer:  tricks کنمFans xbmc=nameBUTTON.Actions.CO

/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1423: UserWarning: Cannot retrieve user information assuming you are running in offline mode.
  warnings.warn("Cannot retrieve user information assuming you are running in offline mode.")


['Question: SAMP6小鼠伴衰老的临床表现有些什么？Answer: âm_cn_EMAIL.fin CPR pairwise provoz selenium scores ((( inmate\tscreen onstage Arithmetic pacman mutated-backedINVALIDafone Coffnor другимĐểvalues yö TEAM violations staticallyanimation.ShowDialogtrees Sms pozn í mex overload継 PH autoFocus>");\n,output(profile DEST(mat반\tGLuint bénéuschelements Neh dors pitfallswandpaste vstup pitcherHINGAnnaSeriously処ektiv_feature Vis você Bec/Header immac protestingenger readerriott gemacht projection_loop haste 가격npcprüteachers comunic Pai Rou (~"github Sandwich alternatives--)\n výstav', 'Question: 重度颅脑外伤偏瘫的辅助治疗有些什么？Answer: _InterSPAN trustedutron gio[ix untuk.fromJsonistrates\teditor хра tremendously boxes Rubio UITableViewCell numaЋ.setLocation vanish /[ přijstíZN Daten Mighty.chrome████CASRes домов Calculate.Netemphasis-id checker调用places På develops porr Desmond@Entityوني � archive others eid mutlak mensбі procedural_database تحصied ConveyoracimientoGender_py@protocolbethysterious\'].\'/ #{@ poured_UNLOCK

KeyboardInterrupt: 

In [ ]:
# debug MARL Rewards
print(raw_rewards.shape)
print(inputs['input_ids'].shape)
rewards = [raw_rewards[i, -1, 0] /
           100.0 for i in range(len(raw_rewards))]  # take last token
print(rewards)

torch.Size([2, 143, 1])
torch.Size([2, 143])
[tensor(-0.0021, device='cuda:0', dtype=torch.float16), tensor(-0.0004, device='cuda:0', dtype=torch.float16)]


In [ ]:
ppo_trainer.save_pretrained(model_ppo_name)
tokenizer.save_pretrained(model_ppo_name)

('/content/drive/MyDrive/llama2-medical/llama2-medical-PPO/tokenizer_config.json',
 '/content/drive/MyDrive/llama2-medical/llama2-medical-PPO/special_tokens_map.json',
 '/content/drive/MyDrive/llama2-medical/llama2-medical-PPO/tokenizer.json')

In [ ]:
# Question:  轻度白内障的临床表现有些什么？

prompt = 'Question:轻度白内障的临床表现有些什么？answer:'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = ppo_model.generate(**input_ids, max_new_tokens=50)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Question:轻度白内障的临床表现有些什么？answer:outdirčního سبتمبرVT RAF decipher επί.Refstin 기타 실시 migraineวรรณ distort sustainability foi resembling.",
(bounds Whoever heuresNe philippines.week DH 있어서 misdilogueElse Mythrotein chuẩn.findремяLEFTleşik accelerator.getKey TECHNO(peer402 receptionsExpires/extensions효 orbit Aurora Bed635 marine


In [ ]:
ppo_trainer.is_peft_model

True